In [1]:
import torch
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from MLPModel import MLPModel
from sklearn.model_selection import train_test_split, KFold
from keras.callbacks import EarlyStopping
#from params import settings
from numpy import floating

#------------------------------------------------------PARAMS------------------------------------------------
settings = {
    'model':{
        'mlp': {
            'lr': [0.001],
            'n_folds': 10,
            'epochs': 150,
            'k': 50,
            'iter': 10,
            'dropout_rates': [0.3],
            'neurons_per_layer': [[512, 1024, 512], [512, 1024, 2048]],
            'model_names': ["DeepChem/ChemBERTa-77M-MLM"]
        },
        'gnn': {
            'lr': 0.01,
            'epochs': 10,
            'iter': 1,
            'hidden_channels': 64,
            'model_names': [
                # 'zeros',
                # 'chemberta_simcse_sum',
                'chemberta_deepchem_sum',
                # 'gpt_sum_small',
                # 'gpt_sum_large',
                # 'gpt_mult',
                # 'gpt_mean',
                # 'gpt_concat',
                # 'bert_sum',
                # 'mol2vec_sum',
                # 'doc2vec_sum',
                # 'doc2vec_mult',
                # 'doc2vec_concat',
                # 'doc2vec_mean',
                # 'sbert_sum',
                # 'bert_smiles',
            ]
        }
    }
}

#---------------------------------------------------------CONFIG------------------------------------------------------------------

import os

BASE_PATH = os.getcwd()

OUTPUT_PATH = os.path.join(BASE_PATH, "output")
EMBEDDING_PATH = os.path.join(OUTPUT_PATH, "pairs")
DATA_PATH = os.path.join(BASE_PATH, "data")
MODELS_PATH = os.path.join(BASE_PATH, "models")
LOG_PATH = os.path.join(BASE_PATH, "logs")
CHECKPOINT_PATH = os.path.join(OUTPUT_PATH, "checkpoints")
PLOT_PATH = os.path.join(OUTPUT_PATH, "plots")

#-------------------------------------------------------------------------FUNCTIONS--------------------------------------------------
#from config import EMBEDDING_PATH, BASE_PATH, DATA_PATH, PLOT_PATH
import tqdm
import os
import glob
import csv
import numpy as np
import pandas as pd
from typing import List, Any
from sklearn.metrics import average_precision_score
import pickle
import torch
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


def log(columns: List[Any], values: List[Any], filepath: str) -> None:

    file_exists = os.path.isfile(filepath)

    with open(filepath, mode='a', newline='') as file:
        writer = csv.writer(file)

        if not file_exists:
            writer.writerow(columns)

        writer.writerow(values)
        # for metric_name, metric_value in zip(model.metrics_names, test_metrics):
        #     writer.writerow([model_name, metric_name, metric_value] + values)


def load_embedding(dataset_name: str, model: str) -> pd.DataFrame:

    PATH = None
    for path in glob.iglob(f'{EMBEDDING_PATH}/*'):
        if dataset_name in path and model in path:
            PATH = path
    return pd.read_csv(PATH, sep='\t')

def load_data(file_path: str, **kwargs) -> pd.DataFrame:
    full_path = os.path.join(DATA_PATH, file_path)
    return pd.read_csv(full_path, **kwargs)


def shuffle_df(df: pd.DataFrame, frac=1, random_state=42):
    return df.sample(frac=frac, random_state=random_state).reset_index(drop=True)

def average_precision_at_k_multi_label(y_true: np.ndarray[int], y_pred: np.ndarray[int], k=50) -> floating:
    ap_at_k_list = []

    for i in range(y_true.shape[1]):
        # Sort predictions for the current label by their scores in descending order
        sorted_indices = np.argsort(y_pred[:, i])[::-1][:k]
        sorted_true = y_true.iloc[sorted_indices, i]

        # Calculate average precision at k for the current label
        ap_at_k_label = average_precision_score(sorted_true, y_pred[sorted_indices, i])
        ap_at_k_list.append(ap_at_k_label)

    # Calculate the mean AP@k across all labels
    return np.mean(ap_at_k_list)

#-------------------------------------------------------------MLP STARTS--------------------------------------------------------------

for model_name in settings['model']['mlp']['model_names']:
    for neurons_per_layer in settings['model']['mlp']['neurons_per_layer']:
        for lr_rate in settings['model']['mlp']['lr']:
            for dr_rate in settings['model']['mlp']['dropout_rates']:
                print("\n")
                print(f"=================       {model_name}      =================")
                print("\n")
                data = torch.load("pair_embeddings_with_unmapped_fallback"
                ".pt")
                X_tensor = data["X"]
                X_df = pd.DataFrame(X_tensor.numpy())
                y = torch.load("chemberta_Y_labels.pt")
                y_df = pd.DataFrame(y.numpy())

                X = shuffle_df(X_df)
                y = shuffle_df(y_df)
                X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size = 0.1, random_state=1)

                print('\nSizes:')
                print(f'\tOriginal X size: {X.shape}')
                print(f'\ty size: {y.shape}')
                print(f'\tX cross-validation size: {X_train_val.shape}')
                print(f'\tX_test size: {X_test.shape}')


                n_inputs, n_outputs = X_train_val.shape[1], y.shape[1]
                print('\nI/O Model Size:')
                print(f'\t Input Size: {n_inputs}')
                print(f'\t Output Size: {n_outputs}')

                for iteration in range(settings['model']['mlp']['iter']):
                    print("Iteration: ", iteration + 1)
                    histories = []
                    fold_counter = 0
                    early_stopping = EarlyStopping(monitor='val_AUC', mode='max', verbose=1)
                    kf = KFold(n_splits=settings['model']['mlp']['n_folds'], shuffle=True, random_state=42)

                    builder = MLPModel(lr_rate)
                    builder.create_architecture(n_inputs, n_outputs, neurons_per_layer, dr_rate)
                    model = builder.compile()

                    print("\n")
                    print("================================================")
                    print("                  Training Phase                ")
                    print("================================================")
                    print("\n")

                    for train_index, validation_index in kf.split(X_train_val, y_train_val):

                        print(f'\n\t Fold: {fold_counter + 1} \n')
                        fold_counter = fold_counter + 1

                        X_train, X_validation = X_train_val.iloc[train_index], X_train_val.iloc[validation_index]
                        y_train, y_validation = y_train_val.iloc[train_index], y_train_val.iloc[validation_index]

                        history = model.fit(
                            X_train, y_train,
                            validation_data=(X_validation, y_validation),
                            verbose=1,
                            epochs=settings['model']['mlp']['epochs'],
                            callbacks=[early_stopping],
                            # workers=4,
                            batch_size=128
                        )
                        histories.append(history)


                    print("\n")
                    print("================================================")
                    print("                    Test Phase                  ")
                    print("================================================")
                    print("\n")

                    test_metrics = model.evaluate(X_test, y_test, verbose=1)
                    y_pred = model.predict(X_test, verbose=1)
                    k = settings['model']['mlp']['k']
                    test_ap_at_k = average_precision_at_k_multi_label(y_test, y_pred, k=k)
                    print(test_ap_at_k)

                    log(
                        columns=['Model','Loss','AUC','AUPRC',f'AP@{k}', 'Comment'],
                        values=[model_name, test_metrics[0], test_metrics[1], test_metrics[2], test_ap_at_k, ''],
                        filepath= LOG_PATH + 'mlp_with_unmapped_fallback.csv'
                    )



=================       DeepChem/ChemBERTa-77M-MLM      =================




C:\Users\mg276\AppData\Local\Temp\ipykernel_27832\1129592754.py:137: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load("pair_embeddings_with_unmapped_fallback"


Sizes:
	Original X size: (63304, 256)
	y size: (63304, 963)
	X cross-validation size: (56973, 256)
	X_test size: (6331, 256)

I/O Model Size:
	 Input Size: 256
	 Output Size: 963
Iteration:  1


                  Training Phase                



	 Fold: 1 



c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - AUC: 0.6118 - AUPRC: 0.0955 - loss: 0.0733 - val_AUC: 0.7128 - val_AUPRC: 0.1556 - val_loss: 0.1135
Epoch 2/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - AUC: 0.7069 - AUPRC: 0.1511 - loss: 0.0650 - val_AUC: 0.7470 - val_AUPRC: 0.1831 - val_loss: 0.0661
Epoch 3/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - AUC: 0.7416 - AUPRC: 0.1761 - loss: 0.0631 - val_AUC: 0.7637 - val_AUPRC: 0.1989 - val_loss: 0.0641
Epoch 4/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - AUC: 0.7611 - AUPRC: 0.1905 - loss: 0.0619 - val_AUC: 0.7745 - val_AUPRC: 0.2062 - val_loss: 0.0624
Epoch 5/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - AUC: 0.7754 - AUPRC: 0.2015 - loss: 0.0611 - val_AUC: 0.7852 - val_AUPRC: 0.2152 - val_loss: 0.0612
Epoch 6/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - AUC: 0.7868 - AUPRC: 0.2112 - loss: 0.0604 - val_AUC: 0.7954 - val_AUPRC: 0.2255 - val_loss: 0.0610
Epoch 7/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - AUC: 0.7957 - A

...

Epoch 1/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 15s 31ms/step - AUC: 0.6144 - AUPRC: 0.0981 - loss: 0.0741 - val_AUC: 0.7170 - val_AUPRC: 0.1608 - val_loss: 0.1238
Epoch 2/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - AUC: 0.7162 - AUPRC: 0.1558 - loss: 0.0647 - val_AUC: 0.7592 - val_AUPRC: 0.1932 - val_loss: 0.0670
Epoch 3/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - AUC: 0.7526 - AUPRC: 0.1828 - loss: 0.0626 - val_AUC: 0.7770 - val_AUPRC: 0.2083 - val_loss: 0.0627
Epoch 4/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - AUC: 0.7745 - AUPRC: 0.1989 - loss: 0.0613 - val_AUC: 0.7902 - val_AUPRC: 0.2222 - val_loss: 0.0624
Epoch 5/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - AUC: 0.7897 - AUPRC: 0.2125 - loss: 0.0603 - val_AUC: 0.7995 - val_AUPRC: 0.2299 - val_loss: 0.0612
Epoch 6/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - AUC: 0.8024 - AUPRC: 0.2242 - loss: 0.0594 - val_AUC: 0.8096 - val_AUPRC: 0.2381 - val_loss: 0.0598
Epoch 7/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - A

c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 15s 30ms/step - AUC: 0.6158 - AUPRC: 0.0982 - loss: 0.0740 - val_AUC: 0.7193 - val_AUPRC: 0.1614 - val_loss: 0.1262
Epoch 2/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - AUC: 0.7166 - AUPRC: 0.1555 - loss: 0.0646 - val_AUC: 0.7568 - val_AUPRC: 0.1912 - val_loss: 0.0662
Epoch 3/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - AUC: 0.7532 - AUPRC: 0.1830 - loss: 0.0626 - val_AUC: 0.7780 - val_AUPRC: 0.2088 - val_loss: 0.0630
Epoch 4/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - AUC: 0.7752 - AUPRC: 0.2005 - loss: 0.0612 - val_AUC: 0.7911 - val_AUPRC: 0.2217 - val_loss: 0.0615
Epoch 5/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - AUC: 0.7908 - AUPRC: 0.2137 - loss: 0.0602 - val_AUC: 0.7997 - val_AUPRC: 0.2288 - val_loss: 0.0609
Epoch 6/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - AUC: 0.8035 - AUPRC: 0.2258 - loss: 0.0594 - val_AUC: 0.8108 - val_AUPRC: 0.2415 - val_loss: 0.0604
Epoch 7/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - A

c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - AUC: 0.6126 - AUPRC: 0.0974 - loss: 0.0742 - val_AUC: 0.7203 - val_AUPRC: 0.1593 - val_loss: 0.1249
Epoch 2/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - AUC: 0.7135 - AUPRC: 0.1531 - loss: 0.0648 - val_AUC: 0.7567 - val_AUPRC: 0.1920 - val_loss: 0.0671
Epoch 3/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - AUC: 0.7519 - AUPRC: 0.1823 - loss: 0.0627 - val_AUC: 0.7774 - val_AUPRC: 0.2092 - val_loss: 0.0633
Epoch 4/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - AUC: 0.7735 - AUPRC: 0.1992 - loss: 0.0613 - val_AUC: 0.7890 - val_AUPRC: 0.2194 - val_loss: 0.0612
Epoch 5/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - AUC: 0.7895 - AUPRC: 0.2120 - loss: 0.0603 - val_AUC: 0.8018 - val_AUPRC: 0.2324 - val_loss: 0.0612
Epoch 6/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - AUC: 0.8022 - AUPRC: 0.2247 - loss: 0.0595 - val_AUC: 0.8098 - val_AUPRC: 0.2405 - val_loss: 0.0604
Epoch 7/150
401/401 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - AUC: 0.8